# Phase 2: Train SLM Ensemble (QLoRA)

**NBME Pipeline — NBME Score Clinical Patient Notes**

Sequentially fine-tunes 3 SLMs using QLoRA:
1. `Qwen/Qwen3.5-4B` (FP16)
2. `google/gemma-4-E2B-it` (BF16)
3. `google/gemma-4-E4B-it` (BF16)

**Hardware**: Single T4 16 GB GPU  **Runtime**: ~60–90 min per model  
**Output**: `adapters/` with 3 LoRA adapters

> Colab tip: Set `MAX_STEPS=500` and `AUG_SAMPLE_RATIO=0.05` to stay within 90-min limit.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
!pip install -q \
  "transformers>=5.5.0" \
  "trl>=1.0.0" \
  "peft>=0.15.0" \
  "bitsandbytes>=0.49.0" \
  "datasets" \
  "accelerate" \
  "scikit-learn" \
  "pandas" "numpy" "packaging"

print("Installation complete ✓")

In [ ]:
# ── Hugging Face login ────────────────────────────────────────────────────────
# Required for Qwen3.5-4B, gemma-4-E2B-it, gemma-4-E4B-it
# Accept licenses: https://huggingface.co/google/gemma-4-E2B-it
#                  https://huggingface.co/google/gemma-4-E4B-it
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# ── Verify data files ────────────────────────────────────────────────────────
import os
for f in ['features.csv', 'patient_notes.csv', 'train.csv']:
    status = '✓' if os.path.exists(f) else '✗ MISSING'
    print(f'  {f}: {status}')
plat = '✓' if os.path.exists('augmented_train.csv') else '⚠ not found (train-only mode)'
print(f'  augmented_train.csv: {plat}')

## Imports, Configuration & Model Registry

In [ ]:
# === IMPORTS ===
import ast
import gc
import json
import logging
import sys
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from sklearn.model_selection import GroupKFold
from transformers import (
    AutoModelForCausalLM,
    AutoModelForImageTextToText,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

# === CONFIGURATION ===
CONFIG = {
    # ── Paths ─────────────────────────────────────────────────────────────────
    "DATA_DIR":          Path("."),
    "ADAPTER_ROOT":      Path("./adapters"),

    # ── Reproducibility ───────────────────────────────────────────────────────
    "SEED": 42,

    # ── Data ──────────────────────────────────────────────────────────────────
    # Fraction of augmented_train.csv to include (1.0 = all ~140k rows).
    # Set lower (e.g., 0.15) for Colab Free Tier to stay within 90-min limit.
    "AUG_SAMPLE_RATIO": 0.15,

    # GroupKFold settings — 10 unique case_nums → 5 folds of 2 cases each
    "N_FOLDS":     5,
    "VAL_FOLD":    4,        # fold 4 = held-out validation cases

    # ── QLoRA LoRA adapter ────────────────────────────────────────────────────
    "LORA_R":       16,
    "LORA_ALPHA":   32,
    "LORA_DROPOUT": 0.05,
    # Explicit target modules — same layer names across Qwen3.5 & Gemma 4
    "LORA_TARGET_MODULES": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],

    # ── Training hyper-params ─────────────────────────────────────────────────
    "PER_DEVICE_BATCH_SIZE":  2,
    "GRADIENT_ACCUMULATION":  4,      # effective batch = 8
    "LEARNING_RATE":          2e-4,
    "NUM_TRAIN_EPOCHS":       2,
    "MAX_SEQ_LENGTH":         512,    # notes ~900 chars ≈ 250-400 tokens + prompt
    "WARMUP_RATIO":           0.05,
    "LR_SCHEDULER":           "cosine",
    "WEIGHT_DECAY":           0.01,
    "LOGGING_STEPS":          50,
    "SAVE_STEPS":             500,
    "EVAL_STEPS":             500,
    "EVAL_STRATEGY":          "steps",
    "SAVE_TOTAL_LIMIT":       1,
    # Set to an integer (e.g., 1000) to cap training steps for Colab speed tests
    "MAX_STEPS":              -1,     # -1 = use NUM_TRAIN_EPOCHS (no step cap)

    # ── T4 VRAM management ────────────────────────────────────────────────────
    "GPU_MEM_UTIL": 0.85,
}

# ── Model registry ────────────────────────────────────────────────────────────
# Each entry fully specifies how to load, quantize, and train one SLM.
#
# compute_dtype notes:
#   • Qwen3.5-4B  → float16  (T4 has FP16 tensor cores; no overflow risk)
#   • Gemma 4     → bfloat16 (fp16 overflows in Gemma4AudioAttention masked_fill)
#
# model_class notes:
#   • Gemma 4 E2B/E4B are multimodal → AutoModelForImageTextToText
#     We train TEXT-ONLY (no pixel_values) — the image encoder path is simply unused.
MODEL_REGISTRY = [
    {
        "name":         "qwen_35_4b",
        "model_id":     "Qwen/Qwen3.5-4B",          # no "-Instruct" suffix in Qwen3.5
        "model_class":  "causal_lm",                  # AutoModelForCausalLM
        "compute_dtype": torch.float16,
        "fp16":          True,
        "bf16":          False,
        "adapter_dir":  Path("./adapters/qwen_35_4b_adapter"),
    },
    {
        "name":         "gemma_4_e2b",
        "model_id":     "google/gemma-4-E2B-it",
        "model_class":  "image_text_to_text",          # AutoModelForImageTextToText
        "compute_dtype": torch.bfloat16,               # REQUIRED — avoid fp16 overflow
        "fp16":          False,
        "bf16":          True,
        "adapter_dir":  Path("./adapters/gemma_4_e2b_adapter"),
    },
    {
        "name":         "gemma_4_e4b",
        "model_id":     "google/gemma-4-E4B-it",
        "model_class":  "image_text_to_text",
        "compute_dtype": torch.bfloat16,
        "fp16":          False,
        "bf16":          True,
        "adapter_dir":  Path("./adapters/gemma_4_e4b_adapter"),
    },
]

# Shared system prompt — identical to Phase 1 for training/inference consistency
SYSTEM_PROMPT = (
    "You are a clinical NLP specialist. "
    "Given a patient note and a clinical feature, extract the EXACT verbatim text spans "
    "from the note that express that feature. "
    "Rules:\n"
    "  1. Copy text character-for-character — do NOT paraphrase.\n"
    "  2. If the feature is absent from the note, return an empty list.\n"
    "  3. Output ONLY valid JSON — no markdown, no explanation.\n"
    "Output format: {\"spans\": [\"exact text 1\", \"exact text 2\"]}"
)

# === LOGGING ===
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)

print('Configuration loaded ✓')

## SECTION 1 — DATA LOADING & PREPARATION

In [ ]:
def safe_parse_list(val) -> list:
    """Parse a Python repr-string list (as stored in train.csv) to an actual list."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    try:
        result = ast.literal_eval(str(val))
        return result if isinstance(result, list) else []
    except (ValueError, SyntaxError):
        return []


def build_assistant_response(annotation_list: list) -> str:
    """
    Convert a list of annotation texts to the JSON string the assistant should output.

    Examples
    --------
    ['chest pain', 'chest tightness']  →  '{"spans": ["chest pain", "chest tightness"]}'
    ['']                               →  '{"spans": []}'
    []                                 →  '{"spans": []}'
    """
    clean_spans = [s.strip() for s in annotation_list if isinstance(s, str) and s.strip()]
    return json.dumps({"spans": clean_spans}, ensure_ascii=False)


def load_and_merge_data(cfg: dict) -> pd.DataFrame:
    """
    Load train.csv + augmented_train.csv, optionally sample augmented data,
    and join with patient_notes + features for the full pn_history / feature_text.

    Returns
    -------
    merged_df : pd.DataFrame
        Columns: pn_num, case_num, feature_num, pn_history, feature_text,
                 annotation_list, assistant_target
    """
    data_dir = cfg["DATA_DIR"]

    # ── Load CSVs ─────────────────────────────────────────────────────────────
    log.info("Loading CSVs …")
    train_df    = pd.read_csv(data_dir / "train.csv")
    pn_df       = pd.read_csv(data_dir / "patient_notes.csv")
    features_df = pd.read_csv(data_dir / "features.csv")

    # Parse annotation lists
    train_df["annotation"] = train_df["annotation"].apply(safe_parse_list)

    # ── Load augmented data if it exists ───────────────────────────────────────
    aug_path = data_dir / "augmented_train.csv"
    if aug_path.exists():
        aug_df = pd.read_csv(aug_path)
        aug_df["annotation"] = aug_df["annotation"].apply(safe_parse_list)

        # Optionally subsample augmented data to stay within Colab time limits
        ratio = cfg["AUG_SAMPLE_RATIO"]
        if ratio < 1.0:
            original_len = len(aug_df)
            n_sample = max(1, int(original_len * ratio))
            aug_df = aug_df.sample(n=n_sample, random_state=cfg["SEED"])
            log.info(
                f"Augmented data sampled: {n_sample} / {original_len} "
                f"({100*ratio:.0f}%)"
            )

        combined = pd.concat([train_df, aug_df], ignore_index=True)
        log.info(f"Combined rows: {len(train_df)} (train) + {len(aug_df)} (augmented) "
                 f"= {len(combined)}")
    else:
        log.warning("augmented_train.csv not found — training on train.csv only (no augmented data).")
        combined = train_df.copy()

    # ── Build lookup maps ─────────────────────────────────────────────────────
    pn_map   = pn_df.set_index("pn_num")["pn_history"].to_dict()
    feat_map = (
        features_df
        .set_index(["case_num", "feature_num"])["feature_text"]
        .to_dict()
    )

    # ── Enrich with full text fields ──────────────────────────────────────────
    combined["pn_history"]    = combined["pn_num"].map(pn_map).fillna("")
    combined["feature_text"]  = combined.apply(
        lambda r: feat_map.get((r["case_num"], r["feature_num"]), ""), axis=1
    )
    combined["assistant_target"] = combined["annotation"].apply(build_assistant_response)

    # Drop rows with empty note or feature
    before = len(combined)
    combined = combined[
        combined["pn_history"].str.strip().ne("") &
        combined["feature_text"].str.strip().ne("")
    ].reset_index(drop=True)
    log.info(f"Dropped {before - len(combined)} rows with empty note/feature. "
             f"Remaining: {len(combined)}")

    return combined[
        ["pn_num", "case_num", "feature_num", "pn_history",
         "feature_text", "annotation", "assistant_target"]
    ]

## SECTION 2 — GROUP K-FOLD SPLIT

In [ ]:
def make_train_val_datasets(
    df:        pd.DataFrame,
    cfg:       dict,
) -> tuple:
    """
    GroupKFold by case_num — ensures zero data leakage between clinical cases.

    With 10 unique case_nums and n_splits=5:
      • Each fold holds 2 cases
      • Train on folds 0-3 (8 cases), validate on fold 4 (2 cases)

    Returns HuggingFace Dataset objects.
    """
    gkf     = GroupKFold(n_splits=cfg["N_FOLDS"])
    groups  = df["case_num"].values
    # X only needs to be the row indices
    X       = np.arange(len(df))

    train_idx, val_idx = None, None
    for fold, (tr_idx, vl_idx) in enumerate(gkf.split(X, groups=groups)):
        if fold == cfg["VAL_FOLD"]:
            train_idx = tr_idx
            val_idx   = vl_idx
            break

    # If VAL_FOLD is the LAST fold, all earlier folds are train
    if train_idx is None:
        # Collect all indices NOT in the val fold
        all_idx = set(range(len(df)))
        val_idx = val_idx
        train_idx = np.array(sorted(all_idx - set(val_idx)))

    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df   = df.iloc[val_idx].reset_index(drop=True)

    log.info(
        f"GroupKFold split — "
        f"train: {len(train_df)} rows (cases {sorted(train_df['case_num'].unique())}) | "
        f"val: {len(val_df)} rows (cases {sorted(val_df['case_num'].unique())})"
    )

    return Dataset.from_pandas(train_df), Dataset.from_pandas(val_df)

## SECTION 3 — PROMPT FORMATTING

In [ ]:
def make_formatting_func(tokenizer):
    """
    Returns a formatting_func compatible with SFTTrainer.

    Converts each dataset row into a FULL conversation string using the model's
    chat template. Loss is computed over the full sequence (input + output).

    This approach is version-agnostic — it does not rely on specific SFTConfig
    parameters like assistant_only_loss that vary between TRL versions.
    """
    def formatting_func(examples: dict) -> list:
        """
        Receives a batch dict; returns a list of formatted strings.
        """
        output_texts = []
        batch_size   = len(examples["pn_history"])

        for i in range(batch_size):
            pn_history    = examples["pn_history"][i] or ""
            feature_text  = examples["feature_text"][i] or ""
            asst_response = examples["assistant_target"][i] or '{"spans": []}'

            messages = [
                {
                    "role":    "system",
                    "content": SYSTEM_PROMPT,
                },
                {
                    "role":    "user",
                    "content": (
                        f"Note: \"{pn_history.strip()}\"\n"
                        f"Feature: {feature_text}\n\n"
                        f"/no_think"   # disable Qwen3.5 chain-of-thought thinking mode
                    ),
                },
                {
                    "role":    "assistant",
                    "content": asst_response,
                },
            ]

            # apply_chat_template converts the messages to the model's exact string format
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            output_texts.append(text)

        return output_texts

    return formatting_func

## SECTION 4 — MODEL & TOKENIZER LOADING

In [ ]:
def build_bnb_config(compute_dtype: torch.dtype) -> BitsAndBytesConfig:
    """
    4-bit NF4 QLoRA quantization config.

    compute_dtype:
      • float16  for Qwen3.5   (T4 native FP16 tensor cores)
      • bfloat16 for Gemma 4   (avoids fp16 overflow in Gemma4 attention layers)
    """
    return BitsAndBytesConfig(
        load_in_4bit             = True,
        bnb_4bit_quant_type      = "nf4",          # NormalFloat4 — standard QLoRA
        bnb_4bit_compute_dtype   = compute_dtype,
        bnb_4bit_use_double_quant= True,           # nested quantisation: ~0.37 bpp saving
        bnb_4bit_quant_storage   = compute_dtype,  # required for FSDP / multi-GPU
    )


def load_model_and_tokenizer(model_spec: dict, bnb_config: BitsAndBytesConfig):
    """
    Load the quantized base model and its tokenizer.

    Handles two model classes:
      • "causal_lm"          → AutoModelForCausalLM     (Qwen3.5)
      • "image_text_to_text" → AutoModelForImageTextToText (Gemma 4)

    For Gemma 4 we load with AutoModelForImageTextToText but use AutoTokenizer —
    all training examples are TEXT-ONLY so pixel_values are never passed.
    """
    model_id    = model_spec["model_id"]
    model_class = model_spec["model_class"]

    log.info(f"Loading model: {model_id}  (class={model_class}) …")

    # ── Common kwargs ─────────────────────────────────────────────────────────
    load_kwargs = dict(
        pretrained_model_name_or_path = model_id,
        quantization_config           = bnb_config,
        torch_dtype                   = model_spec["compute_dtype"],
        device_map                    = "auto",
        # trust_remote_code=False: both Qwen3.5 and Gemma 4 use native HF code
    )

    # ── Model class dispatch ──────────────────────────────────────────────────
    if model_class == "causal_lm":
        model = AutoModelForCausalLM.from_pretrained(**load_kwargs)
    elif model_class == "image_text_to_text":
        model = AutoModelForImageTextToText.from_pretrained(**load_kwargs)
    else:
        raise ValueError(f"Unknown model_class: {model_class!r}")

    # ── Tokenizer ─────────────────────────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        use_fast = True,
    )

    # Ensure pad_token is set (required by SFTTrainer's data collator)
    if tokenizer.pad_token is None:
        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
        log.info("  pad_token set to eos_token")

    # Left-padding: important for causal LM batched training
    tokenizer.padding_side = "right"

    log.info(
        f"  vocab_size={tokenizer.vocab_size}  "
        f"  pad_token='{tokenizer.pad_token}'  "
        f"  eos_token='{tokenizer.eos_token}'"
    )
    return model, tokenizer

## SECTION 5 — LORA ADAPTER SETUP

In [ ]:
def build_lora_config(cfg: dict) -> LoraConfig:
    """
    LoRA configuration — rank 16, alpha 32, targeting all 7 projection layers.

    Using CAUSAL_LM task type for all models:
      • For Qwen3.5: correct — it's a causal decoder
      • For Gemma 4: the language generation head is still causal — PEFT uses
        this to correctly set up gradient flow for the generation objective
    """
    return LoraConfig(
        r               = cfg["LORA_R"],
        lora_alpha      = cfg["LORA_ALPHA"],
        target_modules  = cfg["LORA_TARGET_MODULES"],
        lora_dropout    = cfg["LORA_DROPOUT"],
        bias            = "none",
        task_type       = TaskType.CAUSAL_LM,
    )


def apply_lora(model, lora_config: LoraConfig):
    """
    1. prepare_model_for_kbit_training: enables gradient checkpointing, casts
       LayerNorm to float32, and ensures input embeddings allow gradients.
    2. get_peft_model: wraps the model with LoRA adapter layers.
    """
    # Step 1: prepare for k-bit (4-bit) training
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing = True,
    )

    # Step 2: inject LoRA layers
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    return model

## SECTION 6 — SFT CONFIG

In [ ]:
def build_sft_config(
    model_spec: dict,
    adapter_dir: Path,
    cfg:         dict,
) -> SFTConfig:
    """
    Build an SFTConfig for the given model.

    NOTE — TRL v1.x API:
      • SFT-specific params (max_seq_length, packing, dataset_text_field)
        MUST go in SFTConfig, NOT in SFTTrainer constructor.
      • TrainingArguments params (lr, epochs, batch_size, etc.) also go here
        since SFTConfig is a subclass of TrainingArguments.
    """
    return SFTConfig(
        # ── Output ───────────────────────────────────────────────────────────
        output_dir = str(adapter_dir / "checkpoints"),

        # ── SFT-specific ─────────────────────────────────────────────────────
        max_seq_length   = cfg["MAX_SEQ_LENGTH"],
        packing          = False,   # disable sequence packing for simplicity / stability

        # ── Training schedule ─────────────────────────────────────────────────
        num_train_epochs             = cfg["NUM_TRAIN_EPOCHS"],
        max_steps                    = cfg["MAX_STEPS"],  # -1 = epoch-based
        per_device_train_batch_size  = cfg["PER_DEVICE_BATCH_SIZE"],
        per_device_eval_batch_size   = cfg["PER_DEVICE_BATCH_SIZE"],
        gradient_accumulation_steps  = cfg["GRADIENT_ACCUMULATION"],
        gradient_checkpointing       = True,

        # ── Optimiser ─────────────────────────────────────────────────────────
        learning_rate   = cfg["LEARNING_RATE"],
        weight_decay    = cfg["WEIGHT_DECAY"],
        warmup_ratio    = cfg["WARMUP_RATIO"],
        lr_scheduler_type = cfg["LR_SCHEDULER"],
        optim           = "adamw_8bit",   # paged 8-bit Adam — saves VRAM on T4

        # ── Mixed precision ────────────────────────────────────────────────────
        # fp16 for Qwen (T4 native FP16 cores), bf16 for Gemma (overflow safety)
        fp16 = model_spec["fp16"],
        bf16 = model_spec["bf16"],

        # ── Evaluation & checkpointing ────────────────────────────────────────
        eval_strategy           = cfg["EVAL_STRATEGY"],
        eval_steps              = cfg["EVAL_STEPS"],
        save_strategy           = "steps",
        save_steps              = cfg["SAVE_STEPS"],
        save_total_limit        = cfg["SAVE_TOTAL_LIMIT"],
        load_best_model_at_end  = False,  # we save adapters separately; don't reload

        # ── Logging ───────────────────────────────────────────────────────────
        logging_steps    = cfg["LOGGING_STEPS"],
        logging_dir      = str(adapter_dir / "logs"),
        report_to        = "none",       # disable W&B / TensorBoard on Colab Free

        # ── Reproducibility ───────────────────────────────────────────────────
        seed              = cfg["SEED"],
        data_seed         = cfg["SEED"],

        # ── Misc ──────────────────────────────────────────────────────────────
        remove_unused_columns    = False,   # keep all df columns for formatting_func
        dataloader_num_workers   = 0,       # T4 Colab: 0 avoids multiprocess issues
        group_by_length          = True,    # batch similar-length sequences → less padding
    )

## SECTION 7 — TRAINING PIPELINE FOR ONE MODEL

In [ ]:
def train_one_model(
    model_spec:   dict,
    train_dataset: Dataset,
    val_dataset:   Dataset,
    cfg:           dict,
) -> None:
    """
    Full QLoRA fine-tuning pipeline for a single SLM.

    Steps:
      1. Check for existing adapter (skip if already trained)
      2. Load quantized model + tokenizer
      3. Apply QLoRA (prepare + get_peft_model)
      4. Build formatting function & SFTTrainer
      5. Train
      6. Save LoRA adapters ONLY (not the full model)
      7. VRAM cleanup
    """
    adapter_dir = model_spec["adapter_dir"]
    model_name  = model_spec["name"]

    # ── Skip if adapter already exists ───────────────────────────────────────
    adapter_config_path = adapter_dir / "adapter_config.json"
    if adapter_config_path.exists():
        log.info(f"[{model_name}] Adapter already exists at {adapter_dir} — SKIPPING.")
        return

    adapter_dir.mkdir(parents=True, exist_ok=True)
    log.info("=" * 65)
    log.info(f"  Training: {model_name}  ({model_spec['model_id']})")
    log.info("=" * 65)

    try:
        # ── Step 1: Load model + tokenizer ───────────────────────────────────
        bnb_config  = build_bnb_config(model_spec["compute_dtype"])
        model, tokenizer = load_model_and_tokenizer(model_spec, bnb_config)

        # ── Step 2: Inject LoRA adapters ─────────────────────────────────────
        lora_config = build_lora_config(cfg)
        model       = apply_lora(model, lora_config)

        # ── Step 3: Build formatting function ────────────────────────────────
        fmt_func = make_formatting_func(tokenizer)

        # ── Step 4: Build SFTConfig ───────────────────────────────────────────
        sft_config = build_sft_config(
            model_spec  = model_spec,
            adapter_dir = adapter_dir,
            cfg         = cfg,
        )

        # ── Step 5: Build SFTTrainer ──────────────────────────────────────────
        # TRL v1.x API:
        #   • processing_class= (not tokenizer=) — renamed in v0.16, removed in v1.0
        #   • formatting_func= passed to SFTTrainer constructor (NOT SFTConfig)
        #   • peft_config= omitted — we already called get_peft_model above
        trainer = SFTTrainer(
            model             = model,
            processing_class  = tokenizer,      # ← new API (TRL v1.x)
            args              = sft_config,
            train_dataset     = train_dataset,
            eval_dataset      = val_dataset,
            formatting_func   = fmt_func,       # ← passed here, not in SFTConfig
        )

        # ── Step 6: Train ─────────────────────────────────────────────────────
        log.info(f"[{model_name}] Starting training …")
        train_result = trainer.train()
        log.info(
            f"[{model_name}] Training complete. "
            f"Loss={train_result.training_loss:.4f}  "
            f"Steps={train_result.global_step}"
        )

        # ── Step 7: Save LoRA adapter ONLY ───────────────────────────────────
        # model.save_pretrained() on a PeftModel saves ONLY the adapter weights,
        # NOT the full base model — exactly what we want for Phase 3 inference.
        log.info(f"[{model_name}] Saving LoRA adapter to {adapter_dir} …")
        model.save_pretrained(str(adapter_dir))
        tokenizer.save_pretrained(str(adapter_dir))
        log.info(f"[{model_name}] Adapter saved ✓")

        # Persist training metrics alongside adapter
        metrics = {
            "model_name":      model_name,
            "model_id":        model_spec["model_id"],
            "training_loss":   train_result.training_loss,
            "global_step":     train_result.global_step,
            "train_samples":   len(train_dataset),
            "val_samples":     len(val_dataset),
        }
        with open(adapter_dir / "training_metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)

    finally:
        # ── Step 8: AGGRESSIVE VRAM CLEANUP ──────────────────────────────────
        # CRITICAL: Must free GPU memory before loading the next model.
        # Failure to do so will cause OOM on subsequent model loads.
        log.info(f"[{model_name}] Cleaning up VRAM …")
        try: del model        # noqa: F821
        except NameError: pass
        try: del trainer      # noqa: F821
        except NameError: pass
        try: del tokenizer    # noqa: F821
        except NameError: pass
        try: del fmt_func     # noqa: F821
        except NameError: pass
        try: del sft_config   # noqa: F821
        except NameError: pass
        try: del lora_config  # noqa: F821
        except NameError: pass
        try: del bnb_config   # noqa: F821
        except NameError: pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            mem_free = torch.cuda.mem_get_info()[0] / 1024**3
            mem_total = torch.cuda.mem_get_info()[1] / 1024**3
            log.info(
                f"[{model_name}] VRAM after cleanup: "
                f"{mem_free:.1f} GB free / {mem_total:.1f} GB total"
            )

## Run Phase 2 — Sequential QLoRA Training

In [ ]:
def main():
    cfg = CONFIG
    set_seed(cfg["SEED"])

    log.info("=" * 65)
    log.info("  PHASE 2: SLM Ensemble QLoRA Training")
    log.info("=" * 65)

    # ── Verify transformers version (Gemma 4 requires >= 5.5.0) ──────────────
    import transformers
    from packaging.version import Version
    tf_ver = transformers.__version__
    if Version(tf_ver) < Version("5.5.0"):
        log.warning(
            f"transformers=={tf_ver} detected. "
            "Gemma 4 support requires >= 5.5.0. "
            "Run: pip install -U transformers"
        )

    # ── Step 1: Load and merge data ───────────────────────────────────────────
    merged_df = load_and_merge_data(cfg)

    # ── Step 2: GroupKFold split ──────────────────────────────────────────────
    train_dataset, val_dataset = make_train_val_datasets(merged_df, cfg)
    log.info(f"Train dataset: {len(train_dataset)} examples")
    log.info(f"Val dataset:   {len(val_dataset)} examples")

    # ── Step 3: Create adapter output root ───────────────────────────────────
    cfg["ADAPTER_ROOT"].mkdir(parents=True, exist_ok=True)

    # ── Step 4: Sequential training of all 3 SLMs ────────────────────────────
    for i, model_spec in enumerate(MODEL_REGISTRY):
        log.info(
            f"\n{'='*65}\n"
            f"  Model {i+1}/{len(MODEL_REGISTRY)}: {model_spec['name']}\n"
            f"{'='*65}"
        )

        # Convert Path objects in model_spec to resolved absolute paths
        model_spec["adapter_dir"] = Path(model_spec["adapter_dir"]).resolve()

        train_one_model(
            model_spec    = model_spec,
            train_dataset = train_dataset,
            val_dataset   = val_dataset,
            cfg           = cfg,
        )

        # Extra cleanup between models (belt and suspenders)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ── Summary ───────────────────────────────────────────────────────────────
    log.info("\n" + "=" * 65)
    log.info("  Phase 2 complete — Adapter summary:")
    for model_spec in MODEL_REGISTRY:
        ad = model_spec["adapter_dir"]
        exists = (Path(ad) / "adapter_config.json").exists()
        status = "✓ saved" if exists else "✗ missing"
        log.info(f"    {model_spec['name']:25s}  {status}  → {ad}")
    log.info("=" * 65)


if __name__ == "__main__":
    main()

main()